# Baseline by project weighted by round (public score=0.1057)

What has been done:
- calculate funding amount from weight
- calculate percentage of funding amount by project
- normalize percentage by funder and quarter
- use above funding amount to predict
- null values are filled with 0.5

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('dataset/dataset.csv')
test_df = pd.read_csv('dataset/test.csv')

# dataset size
print(len(df))
print(len(test_df))

df.head()


20958
4261


,id,project_a,project_b,weight_a,weight_b,total_amount_usd,funder,quarter
0,715,https://github.com/mochajs/mocha,https://github.com/chzyer/readline,0.961581,0.038419,2681,opencollective,2016-04
1,716,https://github.com/mochajs/mocha,https://github.com/gulpjs/gulp,0.992302,0.007698,2598,opencollective,2016-04
2,717,https://github.com/chzyer/readline,https://github.com/gulpjs/gulp,0.837398,0.162602,123,opencollective,2016-04
3,718,https://github.com/chzyer/readline,https://github.com/gulpjs/gulp,0.231595,0.768405,652,opencollective,2016-07
4,719,https://github.com/chzyer/readline,https://github.com/mochajs/mocha,0.078851,0.921149,1915,opencollective,2016-07


In [2]:
# check unique projects (project_a and project_b intersect)

df_nunique = set(df['project_a'].unique().tolist() + df['project_b'].unique().tolist())
test_df_nunique = set(test_df['project_a'].unique().tolist() + test_df['project_b'].unique().tolist())

total_unique = df_nunique.union(test_df_nunique)

print(len(df_nunique))
print(len(test_df_nunique))
print(len(total_unique))


104
82
117


In [3]:
# check if there are any row in test_df that both project_a and project_b are in df
test_df[test_df['project_a'].isin(df_nunique) & test_df['project_b'].isin(df_nunique)]

,id,project_a,project_b,total_amount_usd,funder,quarter
0,20884,https://github.com/rollup/rollup,https://github.com/webpack/webpack,29097,opencollective,2024-01
1,20885,https://github.com/rollup/rollup,https://github.com/babel/babel,26093,opencollective,2024-01
2,20886,https://github.com/rollup/rollup,https://github.com/sindresorhus/type-fest,4460,opencollective,2024-01
3,20887,https://github.com/rollup/rollup,https://github.com/inikulin/parse5,2454,opencollective,2024-01
4,20888,https://github.com/rollup/rollup,https://github.com/zloirock/core-js,13229,opencollective,2024-01
...,...,...,...,...,...,...
4247,25206,https://github.com/ethereum/go-ethereum,https://github.com/chainsafe/lodestar,572742,optimism,2024-10
4249,25208,https://github.com/ethereum/go-ethereum,https://github.com/bluealloy/revm,622843,optimism,2024-10
4251,25210,https://github.com/sigp/lighthouse,https://github.com/chainsafe/lodestar,411176,optimism,2024-10
4253,25212,https://github.com/sigp/lighthouse,https://github.com/bluealloy/revm,461277,optimism,2024-10


In [4]:
# calculate funding amount

df['amount_a'] = df['weight_a'] * df['total_amount_usd']
df['amount_b'] = df['weight_b'] * df['total_amount_usd']

In [5]:
# extract unique project_a and amount_a, project_b and amount_b per quarter and funder, and then unfold

df_a = df.groupby(['quarter', 'funder', 'project_a'])['amount_a'].mean().reset_index()
df_b = df.groupby(['quarter', 'funder', 'project_b'])['amount_b'].mean().reset_index()

# rename project_a and project_b to project
df_a.rename(columns={'project_a': 'project'}, inplace=True)
df_b.rename(columns={'project_b': 'project'}, inplace=True)

# rename amount_a and amount_b to amount
df_a.rename(columns={'amount_a': 'amount'}, inplace=True)
df_b.rename(columns={'amount_b': 'amount'}, inplace=True)

# concat df_a and df_b
df_ab = pd.concat([df_a, df_b]).groupby(['quarter', 'funder', 'project']).mean().reset_index()

df_ab

,quarter,funder,project,amount
0,2016-04,opencollective,https://github.com/chzyer/readline,103.0
1,2016-04,opencollective,https://github.com/gulpjs/gulp,20.0
2,2016-04,opencollective,https://github.com/mochajs/mocha,2578.0
3,2016-07,opencollective,https://github.com/chzyer/readline,151.0
4,2016-07,opencollective,https://github.com/gulpjs/gulp,501.0
...,...,...,...,...
1149,2023-10,opencollective,https://github.com/webdriverio/webdriverio,135.0
1150,2023-10,opencollective,https://github.com/webpack/webpack,25140.0
1151,2023-10,opencollective,https://github.com/webreflection/flatted,711.0
1152,2023-10,opencollective,https://github.com/yarnpkg/yarn,1047.0


In [6]:
# divide amount by total amount got per project
df_ab['pct_amount'] = df_ab['amount'] / df_ab['project'].map(df_ab.groupby(['project']).amount.sum())

# average pct_amount by funder and quarter
df_ab_avg = df_ab.groupby(['funder', 'quarter']).pct_amount.mean()

# normalize amount by funder and quarter
df_ab['scaled_amount'] = df_ab['amount'] * df_ab.apply(lambda row: df_ab_avg.get((row['funder'], row['quarter']), np.nan), axis=1)

In [7]:
# approach by doing one score per project
fd = df_ab.groupby(['project']).scaled_amount.sum()

test_fd = test_df.copy()

# fill amount into test_fd by looking up project_a and project_b
test_fd['score_a'] = test_fd['project_a'].map(fd)
test_fd['score_b'] = test_fd['project_b'].map(fd)

test_fd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4261 entries, 0 to 4260
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                4261 non-null   int64  
 1   project_a         4261 non-null   object 
 2   project_b         4261 non-null   object 
 3   total_amount_usd  4261 non-null   int64  
 4   funder            4261 non-null   object 
 5   quarter           4261 non-null   object 
 6   score_a           4011 non-null   float64
 7   score_b           4081 non-null   float64
dtypes: float64(2), int64(2), object(4)
memory usage: 266.4+ KB


In [8]:
# calculate pred (weight_a by score_a / (score_a + score_b))

test_fd['pred'] = test_fd['score_a'] / (test_fd['score_a'] + test_fd['score_b'])

# fill in null values with 0.5
test_fd['pred'].fillna(0.5, inplace=True)

# save id, pred to csv
test_fd[['id', 'pred']].to_csv('submissions/baseline_by_project_weighted_by_round.csv', index=False)

/var/folders/gt/sg3v8rd13l52jx91mfbgzbfc0000gn/T/ipykernel_98395/235156080.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_fd['pred'].fillna(0.5, inplace=True)


In [9]:
# check against train set

df['pred_amount_a'] = df['project_a'].map(fd)
df['pred_amount_b'] = df['project_b'].map(fd)

df['pred'] = df['pred_amount_a'] / (df['pred_amount_a'] + df['pred_amount_b'])

# calculate MSE between pred and weight_a

df['mse'] = (df['pred'] - df['weight_a']) ** 2

df['mse'].mean()

np.float64(0.07336304058389455)